# Ассистент по лекциям
Проектный практикум. Три модели: **Whisper** (речь → текст), **Qwen** (конспект и ответы), **multilingual-e5** (поиск по смыслу).

Перед запуском: *Среда выполнения → Сменить среду выполнения → T4 GPU*. Затем запускайте ячейки по порядку.

## 0. Установка библиотек

In [ ]:
!pip -q install transformers accelerate sentence-transformers

## 1. Загрузка аудио лекции
Выберите файл (mp3, wav, m4a) в окне загрузки. Он сохранится как `lecture.mp3`.

In [ ]:
from google.colab import files
import os
up = files.upload()
name = next(iter(up))
os.rename(name, "lecture.mp3")
print("Загружено:", name)

Saving f035f883f5c330d.mp3 to f035f883f5c330d.mp3
Загружено: f035f883f5c330d.mp3


## 2. Модель 1: Whisper — речь в текст
Аудио режется на куски по 30 секунд, каждый распознаётся отдельно.

In [ ]:
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=0)
text = asr("lecture.mp3", chunk_length_s=30, generate_kwargs={"language": "russian"})["text"]
print(len(text), "символов")
print(text[:500])

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE 

15874 символов
 Область знаний Лек поговорим про то, как мозг меняется со временем, что с ним происходит отрождение и куда мы идем, что с нами происходит в пожилом возрасте. Если посмотреть на динамику изменений вообще очень издалека, то с точки зрения количества изменений, которые происходят, у нас очень насыщенная жизнь в начале и потом очень мало всего происходит под конец, при этом направление изменений противоположное. Если смотреть, например, на количество связей между нейронами или на количество даже са


## 3. Модель 2: Qwen — конспект
Функция `chat` используется и в конспекте, и в ответах на вопросы. В конспект берутся первые 6000 символов (ограничение, см. документ).

In [ ]:
llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", torch_dtype="auto", device_map="auto")

def chat(prompt, n=500):
    msgs = [{"role": "system", "content": "Ты помощник студента. Отвечай по-русски."},
            {"role": "user", "content": prompt}]
    return llm(msgs, max_new_tokens=n)[0]["generated_text"][-1]["content"]

print(chat(f"Сделай краткий конспект лекции пунктами:\n\n{text[:6000]}"))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


**Конспект лекции: "Строение и функции мозга"**

1. **Минимальный возраст начала образования мозга**: Значительно раньше, чем считается.

2. **Величина и скорость роста мозга**: Растет до 3 лет, затем медленно до 25-30 лет (по 1000-2000 мегавершин).

3. **Максимальное количество нейронов и связей**: До 25-30 лет.

4. **После этого**: Количество нейронов сокращается, а связи между ними также.

5. **Старение мозга**: Происходит через структурные изменения:
   - Уменьшение объема мозга.
   - Признаки старения: уменьшение воды в мозге.
   - Подверженность старению различным областям мозга (например, фронтальной области).
   - Особый интерес к Гиппокампу: его разрушение часто указывает на альцгеймерскую болезнь.

6. **Факторы, увеличивающие старение мозга**: 
   - Положительное влияние социальных связей.
   - Эмоциональная активность (многие исследования показывают, что активное участие в обществе уменьшает риск старения).

7. **Проблемы и лечение**: 
   - Нормальное старение и Альцгеймерск

## 4. Модель 3: E5 — поиск по смыслу и ответ на вопрос
Лекция режется на куски по 500 символов. На вопрос выбираются 3 ближайших по смыслу куска, Qwen отвечает только по ним (подход RAG).

In [ ]:
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer("intfloat/multilingual-e5-small")
chunks = [text[i:i+500] for i in range(0, len(text), 500)]
vecs = emb.encode(["passage: " + c for c in chunks], normalize_embeddings=True)

def ask(q, show=False):
    hits = util.semantic_search(emb.encode("query: " + q, normalize_embeddings=True), vecs, top_k=3)[0]
    found = [chunks[h["corpus_id"]] for h in hits]
    if show:
        for f in found: print("НАЙДЕНО:", f, "\n---")
    ctx = "\n---\n".join(found)
    return chat(f"Ответь на вопрос только по фрагментам лекции.\n\nФрагменты:\n{ctx}\n\nВопрос: {q}", 300)

print(ask("Что такое градиентный спуск?", show=True))

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


НАЙДЕНО: пустим, про человека, просто смотрим на его мозг, например, при помощи структурной МРТ. Мы можем заподозрить, что он старый или даже как в случае с какими-то заболеваниями сказать, что он как-то преждевременно постарел по тому, как он устроен. И это касается как общих параметров мозга. Если мы видим, что мозг уменьшился, то, вероятно, у нас произошли какие-то взрослые изменения. То есть если очень грубо, то со временем, с воздействием, мы можем сказать, что он старый или даже вот как в случае с  
---
НАЙДЕНО: о временем, с возрастом у нас мозг потихонечку усыхает, потихонечку зимается и выражается прямо в том числе, что в нем становится процентно меньше воды. Если смотреть на какие-то отдельные регионы, то некоторые из них в большей степени подвержены именно старению. Это, например, фронтальная региона мозга. Это передняя часть мозга, которая связана с каким-то клиентивными процессами. И это, в принципе, ассоциировано с тем, что мы видим. В пожилом возрасте мы видим, что какие

## 5. Свой вопрос
Замените текст вопроса на тему вашей лекции.

In [ ]:
q = input("Ваш вопрос по лекции: ")
print(ask(q))

Ваш вопрос по лекции: Как избежать скорого старения мозга?


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Чтобы избежать скорого старения мозга, следует сохранять социальные связи и использовать мозг регулярно. Это помогает предотвратить разрушающие процессы, которые могут привести к альцгеймерскому заболеванию или другим проблемам. Не стоит забывать, что мозг постоянно требует активности для сохранения своих функций и структуры.
